In [3]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [4]:
raw = load_dataset('imdb')
df  = pd.DataFrame(raw['train']).sample(5000, random_state=42).reset_index(drop=True)
df.columns = ['text', 'sentiment']
print(df.head())
print(df.shape)

                                                text  sentiment
0  Dumb is as dumb does, in this thoroughly unint...          0
1  I dug out from my garage some old musicals and...          1
2  After watching this movie I was honestly disap...          0
3  This movie was nominated for best picture but ...          1
4  Just like Al Gore shook us up with his painful...          1
(5000, 2)


In [5]:
import re

def clean_txt(t):
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip().lower()

df.dropna(inplace=True)
df['text'] = df['text'].apply(clean_txt)

le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])

texts  = df['text'].tolist()
labels = df['label'].tolist()

In [6]:
X_tr, X_tmp, y_tr, y_tmp = train_test_split(texts, labels, test_size=0.2, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

In [7]:
tok = AutoTokenizer.from_pretrained('bert-base-uncased')

tr_enc  = tok(X_tr,  truncation=True, padding=True, max_length=128)
val_enc = tok(X_val, truncation=True, padding=True, max_length=128)
te_enc  = tok(X_te,  truncation=True, padding=True, max_length=128)

In [8]:
class IMDbDS(torch.utils.data.Dataset):
    def __init__(self, enc, lbls):
        self.enc  = enc
        self.lbls = lbls
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.lbls[i])
        return item
    def __len__(self):
        return len(self.lbls)

tr_ds  = IMDbDS(tr_enc,  y_tr)
val_ds = IMDbDS(val_enc, y_val)
te_ds  = IMDbDS(te_enc,  y_te)

In [11]:
def get_model():
    return AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

def get_args(outdir):
    return TrainingArguments(
        output_dir=outdir,
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=2,
        eval_strategy='epoch',
        save_strategy='epoch',
        report_to='none'
    )

def calc_metrics(ep):
    logits, lbls = ep
    preds = logits.argmax(axis=1)
    return {
        'accuracy':  accuracy_score(lbls, preds),
        'precision': precision_score(lbls, preds),
        'recall':    recall_score(lbls, preds),
        'f1':        f1_score(lbls, preds)
    }

def make_trainer(model, args):
    return Trainer(model=model, args=args, train_dataset=tr_ds,
                   eval_dataset=val_ds, compute_metrics=calc_metrics)

In [ ]:
print('=== Exp 1: Freeze all BERT layers ===')
m1 = get_model()
for p in m1.bert.parameters():
    p.requires_grad = False

t1 = make_trainer(m1, get_args('./exp1'))
t1.train()  

=== Exp 1: Freeze all BERT layers ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\vides\AppData\Local\Programs

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.697112,0.694962,0.498000,0.481069,0.923077,0.632504
2,0.692112,0.689750,0.516000,0.490196,0.854701,0.623053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\vides\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.6946123352050781, metrics={'train_runtime': 1886.5672, 'train_samples_per_second': 4.241, 'train_steps_per_second': 0.53, 'total_flos': 526222110720000.0, 'train_loss': 0.6946123352050781, 'epoch': 2.0})

In [13]:
print('=== Exp 2: Fine-tune last 2 BERT layers ===')
m2 = get_model()
for name, p in m2.bert.named_parameters():
    p.requires_grad = 'encoder.layer.10' in name or 'encoder.layer.11' in name

t2 = make_trainer(m2, get_args('./exp2'))
t2.train()

=== Exp 2: Fine-tune last 2 BERT layers ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\vides\AppData\Local\Programs

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.500658,0.356616,0.854000,0.813230,0.893162,0.851324
2,0.364463,0.327527,0.866000,0.843621,0.876068,0.859539


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\vides\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.4325604095458984, metrics={'train_runtime': 2194.0928, 'train_samples_per_second': 3.646, 'train_steps_per_second': 0.456, 'total_flos': 526222110720000.0, 'train_loss': 0.4325604095458984, 'epoch': 2.0})

In [ ]:
print('=== Exp 3: Full fine-tune ===')
m3 = get_model()
t3 = make_trainer(m3, get_args('./exp3'))
t3.train()

In [ ]:
def eval_test(trainer, name):
    out    = trainer.predict(te_ds)
    y_pred = out.predictions.argmax(axis=1)
    return {
        'Experiment': name,
        'Accuracy':   round(accuracy_score(y_te, y_pred), 4),
        'Precision':  round(precision_score(y_te, y_pred), 4),
        'Recall':     round(recall_score(y_te, y_pred), 4),
        'F1':         round(f1_score(y_te, y_pred), 4),
        'preds':      y_pred
    }

r1 = eval_test(t1, 'Frozen BERT')
r2 = eval_test(t2, 'Last 2 Layers')
r3 = eval_test(t3, 'Full Finetune')

res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'preds'} for r in [r1, r2, r3]])
print(res_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cls_names = ['Negative', 'Positive']

for ax, r in zip(axes, [r1, r2, r3]):
    cm = confusion_matrix(y_te, r['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=cls_names, yticklabels=cls_names, ax=ax)
    ax.set_title(r['Experiment'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
x, w    = np.arange(len(metrics)), 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, r in enumerate([r1, r2, r3]):
    ax.bar(x + i*w, [r[m] for m in metrics], w, label=r['Experiment'])

ax.set_xticks(x + w)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_title('Experiment Comparison')
ax.legend()
plt.tight_layout()
plt.show()

best = max([r1, r2, r3], key=lambda r: r['F1'])
print(f"Best: {best['Experiment']} | F1={best['F1']}")